# 00 Swiss Legal Agent: Project Overview

**Team LexGraph** · FHNW BSc Business Artificial Intelligence · Generative AI & Agent Systems, HS 2026 · Moodle group Team_C

**Coaching:** Sandro Schwander · **Module lead:** Prof. Dr. Andreas Martin

**Final presentation:** Monday, 23 November 2026, 13:00. Ten minutes including a live demo, every team member presents.

---

A lawyer types in a case. A router agent identifies the area of law. A specialist agent pulls the relevant
statutory articles and Federal Supreme Court decisions from our knowledge base and drafts an answer
(applicable norm, subsumption, result) with sources. A reviewer agent checks the draft and sends it back
when something is missing. Uncertain cases go to a human.

This is a prototype built in a university course. It is not legal advice and not an official service.

## 1. Problem and scope

- **Problem.** General-purpose language models answer Swiss legal questions fluently, but they invent article
  numbers, deadlines and court decisions. Notebook 01 shows this live. For a lawyer, a wrong citation is worse
  than no answer at all.
- **Goal.** An agent that answers only from verifiable public sources and cites every article it relies on.
- **Scope.** Employment law (Swiss Code of Obligations, Art. 319 ff. OR) and tenancy law (Art. 253 ff. OR).
- **Sources.** Public data only: Fedlex (consolidated federal law) and published Federal Supreme Court decisions (BGE). No client data.
- **Out of scope.** Other areas of law, advice to non-lawyers, fine-tuning a model.
- **Hard rule from our coach.** No low-code or no-code tools (n8n, Voiceflow and similar). Everything is built in Python and LangGraph.

## 2. Architecture

```
               question
                  │
          ┌───────▼────────┐
          │  Input guards  │  off-topic, jailbreak, redaction of personal data (nDSG)
          └───────┬────────┘
          ┌───────▼────────┐
          │     Router     │  employment law | tenancy law | out of scope
          └───────┬────────┘
        ┌─────────┴──────────┐
┌───────▼────────┐  ┌────────▼───────┐
│ Employment law │  │  Tenancy law   │◄──► MCP server "fedlex-mcp"
│     agent      │  │     agent      │      get_article · search_law · search_decisions
└───────┬────────┘  └────────┬───────┘             │
        └─────────┬──────────┘           knowledge base: FAISS + BM25, reranking
          ┌───────▼────────┐
          │    Reviewer    │  sends the draft back, at most 2 rounds
          └───────┬────────┘
          ┌───────▼────────┐
          │ Citation check │  every cited article must appear in the retrieved sources
          └───────┬────────┘
       confident  │  uncertain ──► human in the loop
                  ▼
        answer with sources
```

**Stack.** LangGraph and LangChain · FHNW LiteLLM proxy (OpenAI-compatible) for commercial and open models ·
Swiss AI Platform for Apertus · Apertus Swiss embeddings · FAISS and BM25 · MCP (FastMCP server,
`langchain-mcp-adapters` client) · Gradio · LangSmith tracing.

## 3. Team and roles

Every building block has exactly one owner who decides and delivers. Supporting means reviewing, pairing and
stepping in when the owner is blocked. Every member must still be able to explain the whole system.

| Member | Role | Owns | Supports |
|---|---|---|---|
| Jack Ondoua | **Product Lead & Knowledge Architect** | Project lead, planning, coaching contact, presentation and submission. Knowledge base and RAG pipeline. Evaluation: test set, LLM-as-a-judge, F2, F4 and analysis of F1. Gradio UI, demo and this Deepnote project. | Agent prompts, citation check test cases |
| Tim Rusch | **Agent Orchestration & Safety Engineer** | LangGraph state, router, specialist agents, reviewer loop, human in the loop. MCP server `fedlex-mcp`. Model integration (LiteLLM, Apertus) and F1 runs. Guardrails and citation check. Tracing, GitHub repository, backup video. | Knowledge base & RAG, evaluation |

Roles were first assigned with a skill check (skills and motivation, weighted per role). Since 12 October 2026 the team
consists of Jack and Tim; the building blocks of two former members were redistributed by strength and preference.

## 4. Research questions

- **F1 Model comparison.** Apertus (Swiss open-weight model) against a commercial model on the FHNW proxy.
  We measure answer quality (judge score), citation accuracy, cost per case and latency.
- **F2 RAG ablation (main experiment, chosen by team vote).** Prompt only, RAG, and RAG through MCP tools.
  We measure Hit@k, citation precision and judge score.
- **F3 Reviewer loop.** Does a second pass by the reviewer improve the answer, and what does it cost?
- **F4 Judge validity.** How closely does the LLM judge agree with our own ratings on about ten cases?

## 5. Notebook map

| # | Notebook | Building block | Owner |
|---|---|---|---|
| 00 | Project Overview | Purpose & scope | Jack |
| 01 | Setup and First Model Call | LLM selection | Tim |
| 02 | Knowledge Base and RAG | Knowledge & RAG | Jack |
| 03 | Tools and MCP Server | Tools & MCP | Tim |
| 04 | Agent Graph | Orchestration | Tim |
| 05 | Guardrails | Safeguarding | Tim |
| 06 | Evaluation | Testing & evaluation | Jack |
| 07 | Demo UI | User interface | Jack |

**Run order.** 01 checks the keys. 02 writes `data/or_articles.jsonl`, which 03 and 04 read.
Every notebook starts with owner, goal and Definition of Done, and ends with a decision log.

## 6. Milestones

| Iteration | Until | Goal |
|---|---|---|
| W3 | 18 Oct | Keys work for everyone, repository and Deepnote ready, one case runs end to end |
| W4 | 25 Oct | Retrieval with Hit@k, MCP server, router with both specialists and reviewer, UI prototype |
| W5 | 1 Nov | Input guards, citation check, LLM judge, tracing |
| W6 | 8 Nov | Experiments F1, F2, F4. Inspect & Adapt on 9 Nov |
| W7 | 15 Nov | Feature freeze, presentation draft, notebooks final |
| W8 | 22 Nov | Backup video of the demo, dress rehearsal |
| | 23 Nov | Final presentation 13:00, submission on Moodle |

## 7. Working agreements

- **Code lives on GitHub** (link follows). Deepnote documents, explains and demonstrates.
- **Secrets never go into code.** Deepnote: *Integrations → Environment variables*, connected to this project.
  Names: `LITELLM_API_KEY`, `SWISSAI_API_KEY`, optional `LANGSMITH_API_KEY`.
- **Versions are pinned** in `requirements.txt`, the same tested set as the course notebooks. Do not bump single packages.
- **Python 3.11 or newer.** Deepnote: *Environment → Python version*.
- **Decision log in every notebook.** Decision, rejected alternative, reason. That is exactly what the oral exam asks: why this and not something else?
- **Practice blocks in class** are our main working time. Acknowledge messages briefly and ask directly when something is unclear.

## 8. Decision log

| Date | Decision | Rejected alternative | Reason |
|---|---|---|---|
| Oct 2026 | Build the agent in LangGraph | Linear chain, low-code tools | Reviewer loop, shared state and human in the loop need a graph. Low-code is not allowed. |
| Oct 2026 | RAG instead of fine-tuning | Fine-tuning Apertus on legal texts | A model learns new facts only in (continued) pre-training. Post-training changes behaviour (lecture 12). RAG stays current and every statement is citable. |
| Oct 2026 | Scope: employment and tenancy law | Several areas of law | Two areas are enough to show routing. More areas would dilute the test set. |
| 11 Oct 2026 | Roles from a weighted skill check | Free choice of roles | Every building block gets a clear owner, strengths are used deliberately. |
| 11 Oct 2026 | Code on GitHub, documentation and demo in Deepnote | Everything inside notebooks | Reviewable code and a clean submission. The notebooks stay readable. |
| 11 Oct 2026 | Project work mainly in the practice blocks in class | Fixed weekly evening slot on site | Sport commitments and long commutes. In class we can ask the coach directly. |
| 12 Oct 2026 | Continue as a team of two with reduced scope: test set of 20 cases, F3 optional, BGE optional, own guardrail rules | Keep the full scope | Two people cannot deliver the full scope in six weeks. One owner per building block keeps responsibilities clear. |

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=9b9d17b7-b4fd-473b-beb2-f7ed180fbdd3' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>